# Retrain Pipeline Debug Log — Lab Notes

**Date:** 2026-07-19
**Goal:** get `POST /api/v1/ml/retrain` to actually work end-to-end, then sanity-check whether the accuracy numbers it produces (96-98%) are trustworthy.

**Short version:** the retrain button had never successfully completed a full run before this session — 9 separate bugs stacked on top of each other, one of which destructively deleted a live model file mid-run. All are fixed below. Once it ran clean, the resulting 96-98% accuracy numbers turned out to be **inflated**, for a specific, explainable reason (see Part 3). This notebook is the audit trail.

## Part 1 — Bugs found and fixed, in the order they were hit

Every one of these was found by actually running the retrain pipeline and reading the real traceback — not by code review alone. Each fix was deployed and re-tested before moving to the next bug.

In [1]:
import pandas as pd

bugs = [
    dict(n=1, area="Frontend auth", symptom="401 Not authenticated on POST /machines",
         root_cause="@metagptx/web-sdk's client.entities.* reads its own token from localStorage['token'], "
                     "independent of the app's injectToken wrapper which only set localStorage['access_token']",
         fix="Login.tsx now writes both localStorage keys on login/register",
         file="app/frontend/src/modules/auth/Login.tsx"),
    dict(n=2, area="Audit logging", symptom="500 error on machine create, but the machine WAS saved (visible after refresh)",
         root_cause="data.model_dump() kept raw datetime objects; SQLAlchemy's JSON column type calls "
                     "json.dumps() on commit \u2192 TypeError on datetime. That failure triggered a db.rollback(), "
                     "which expired the already-committed machine ORM object; the except-block then touched "
                     "result.id on that expired object, triggering a doomed sync lazy-load in an async context "
                     "\u2192 greenlet crash \u2192 500 to the client despite the row being safely in the DB",
         fix="model_dump(mode='json') for audit payloads; except-block uses a pre-captured safe_id string "
             "instead of touching the expired ORM object. Same copy-pasted bug fixed in 5 places "
             "(machines, ordres_intervention, ordres_travail \u00d7 single+batch create)",
         file="app/backend/modules/shared/routes/{machines,ordres_intervention,ordres_travail}/crud.py"),
    dict(n=3, area="Retrain query", symptom="'No new ground truth data to retrain' despite 240 fresh interventions in the DB",
         root_cause="`OrdresIntervention.actual_failure_type is not None` and `not OrdresIntervention.retrained` "
                     "are Python identity/truthiness checks evaluated at query-build time, not SQL predicates. "
                     "A SQLAlchemy Column object is truthy in Python, so `not column` \u2192 False \u2192 the "
                     ".where() clause silently became `WHERE true AND false AND ...` \u2192 always 0 rows",
         fix="Use .is_not(None) / .is_(False) \u2014 real SQL predicates, not Python operators on ORM columns",
         file="app/backend/modules/ml/services/ml_retraining.py"),
    dict(n=4, area="File path", symptom="[Errno 2] No such file or directory: '/app/modules/ai4i2020.csv'",
         root_cause="DATA_PATH walked up 3 directory levels from ml_retraining.py (services/ml/modules), "
                     "landing at /app/modules \u2014 needed a 4th level to reach /app where the CSV actually lives",
         fix="Added the missing os.path.dirname() call",
         file="app/backend/modules/ml/services/ml_retraining.py"),
    dict(n=5, area="Missing features", symptom="\"['temp_delta', 'rpm_torque'] not in index\" then later \"['tool_wear_sq'] not in index\"",
         root_cause="P1/P2/P5/P6 expect engineered features (temp_delta, rpm_torque, tool_wear_sq) that are "
                     "computed at inference time in the ml-microservice, but the retrain pipeline never "
                     "reproduced that computation before handing data to the models",
         fix="Compute the same 3 engineered columns right after building combined_df, using the identical "
             "formulas as ml-microservice/src/core/feature_pipeline.py",
         file="app/backend/modules/ml/services/ml_retraining.py"),
    dict(n=6, area="Docker volume", symptom="[Errno 30] Read-only file system when trying to back up a model before retraining",
         root_cause="docker-compose.yml mounted the shared models volume :ro into the backend container \u2014 "
                     "blocks the exact write the retrain feature needs to do",
         fix="Flipped both model volume mounts from :ro to read-write; recreated the backend container",
         file="docker-compose.yml"),
    dict(n=7, area="DESTRUCTIVE: model file loss", symptom="ml_model_p3_rul.pkl had no live file left, only backups, after two failed retrain attempts",
         root_cause="_backup_model() renames the live pkl to a .bak file BEFORE _fit_model_for_type() runs. "
                     "There was no try/except around that call \u2014 when fit raised an exception (bug #5's "
                     "missing-feature error), the restore-on-failure logic never executed because it lived "
                     "AFTER the unprotected call. The live model was gone with nothing to replace it",
         fix="Restored P3 from its own backup file immediately. Then wrapped _fit_model_for_type() in try/except "
             "that restores the backup on ANY exception before returning a 'failed' status. Also wrapped the "
             "outer per-model loop so one model's failure can't abort the rest of the batch",
         file="app/backend/modules/ml/services/ml_retraining.py"),
    dict(n=8, area="XGBoost feature names", symptom="ValueError: feature_names must be string, and may not contain [, ] or <",
         root_cause="P3's pkl had no 'xgb_features' safe-name mapping stored, so the code fell back to reusing "
                     "the raw column names (e.g. 'Air temperature [K]') as the 'safe' names \u2014 XGBoost "
                     "rejects brackets outright, so the rename was a no-op",
         fix="When xgb_features is missing, derive safe names by stripping [ ] < from the raw feature list "
             "instead of reusing them unchanged",
         file="app/backend/modules/ml/services/ml_retraining.py"),
    dict(n=9, area="Dashboard", symptom="'Pr\u00e9cision Mod\u00e8le P1' card stuck forever on 'Chargement...'",
         root_cause="`client = await ml_client.get_client()` \u2014 get_client() is a synchronous method "
                     "(no async def), so awaiting it raised a TypeError that was silently caught and surfaced "
                     "as a generic error message",
         fix="Removed the incorrect await",
         file="app/backend/core/ml_client.py"),
]
df_bugs = pd.DataFrame(bugs).set_index("n")
pd.set_option("display.max_colwidth", None)
df_bugs[["area", "symptom"]]

,area,symptom
n,,
1,Frontend auth,401 Not authenticated on POST /machines
2,Audit logging,"500 error on machine create, but the machine WAS saved (visible after refresh)"
3,Retrain query,'No new ground truth data to retrain' despite 240 fresh interventions in the DB
4,File path,[Errno 2] No such file or directory: '/app/modules/ai4i2020.csv'
5,Missing features,"""['temp_delta', 'rpm_torque'] not in index"" then later ""['tool_wear_sq'] not in index"""
6,Docker volume,[Errno 30] Read-only file system when trying to back up a model before retraining
7,DESTRUCTIVE: model file loss,"ml_model_p3_rul.pkl had no live file left, only backups, after two failed retrain attempts"
8,XGBoost feature names,"ValueError: feature_names must be string, and may not contain [, ] or <"
9,Dashboard,'Précision Modèle P1' card stuck forever on 'Chargement...'


Two more cosmetic/UX fixes, not correctness bugs, done in the same pass:

- **"Dernier Retraitement" tile was hardcoded** to the literal text "Il y a 2 jours" / "État: Stable" — never wired to real data at all. Now reads `metrics.retrained_at` and computes a real relative time.
- **Model health table had no accuracy column.** Added a "Précision" column driven by a new `_headline_metric()` helper in `model_registry.py` that reads each model's own stored metric (ROC-AUC for P1, F1-macro for P2/P5, R² for P3/P6, F1 for P7) — P4 correctly shows blank, see Part 3.

## Part 2 — First clean run (bugs 1-9 fixed)

With every bug above fixed, the pipeline completed successfully for the first time. Validation was a **random row-level 80/20 split** across all machines pooled together.

In [2]:
run1 = pd.DataFrame([
    dict(model="P1 Failure probability", metric="ROC-AUC", score=0.9809, status="success"),
    dict(model="P2 Failure type",        metric="F1 macro", score=0.6667, status="REJECTED by 0.70 gate"),
    dict(model="P3 RUL estimate",        metric="R\u00b2",       score=0.9604, status="success"),
    dict(model="P4 Anomaly",             metric="n/a (unsupervised)", score=None, status="success, no accuracy concept"),
    dict(model="P5 Priority",            metric="F1 macro", score=0.8140, status="success"),
    dict(model="P6 Schedule",            metric="n/a", score=None, status="skipped, no scheduling ground truth"),
])
run1

,model,metric,score,status
0,P1 Failure probability,ROC-AUC,0.9809,success
1,P2 Failure type,F1 macro,0.6667,REJECTED by 0.70 gate
2,P3 RUL estimate,R²,0.9604,success
3,P4 Anomaly,n/a (unsupervised),NaN,"success, no accuracy concept"
4,P5 Priority,F1 macro,0.8140,success
5,P6 Schedule,n/a,NaN,"skipped, no scheduling ground truth"


**P2 got automatically rejected by its own accuracy gate** (needs \u226570% F1, scored 66.7%) \u2014 the old model was restored, nothing degraded. This is the safety mechanism (bug #7's fix) working exactly as intended.

P1/P3 at 96-98% looked suspiciously good. That's Part 3.

## Part 3 — Why 96-98% is not a real number (and what is)

Two separate problems inflate the score, on top of each other:

**Problem A — near-duplicate rows leak across the split.** The seed data (`seed_ml_data_all.py`) generates each machine's failure cycles as smooth linear interpolations (`_lerp`) with tiny Gaussian noise. Consecutive telemetry rows within one cycle are near-copies of each other. A **random row-level split** throws some of those near-copies into training and their near-twins into validation \u2014 the model isn't generalizing, it's recognizing rows it already effectively saw.

**Problem B \u2014 the label is a hard-coded formula on the model's own input feature.** In `seed_ml_data_all.py::_cycle_params()`:

```python
if expected_wear >= 190: failure_type = "TWF"
elif expected_wear >= 155: failure_type = "OSF"
elif expected_wear >= 100: failure_type = "PWF"
elif expected_wear >= 60:  failure_type = "RNF"
else: failure_type = "NONE"
```

`tool_wear` is both an **input feature** the model reads AND the **direct source of the label** it predicts. The model doesn't need to learn a real pattern \u2014 it just needs to find `wear >= 190`, which XGBoost does almost perfectly by construction. This is the same failure mode as writing an exam where the answer key is a rewording of the question.

### Fix attempted: group-based holdout split

Instead of a random row shuffle, hold out **one entire machine** for validation \u2014 the model trains on machines 1-3 and gets tested on machine 4, which it has never seen a single row of. This directly targets Problem A (no more near-duplicate leakage, since whole cycles move together).

In [3]:
run2 = pd.DataFrame([
    dict(model="P1 Failure probability", metric="ROC-AUC", before=0.9809, after=0.9911, moved="no \u2014 see explanation below"),
    dict(model="P2 Failure type",        metric="F1 macro", before=0.6667, after=0.6884, moved="no, still rejected \u2014 consistent signal"),
    dict(model="P3 RUL estimate",        metric="R\u00b2",       before=0.9604, after=0.6661, moved="YES \u2014 real generalization gap exposed"),
    dict(model="P5 Priority",            metric="F1 macro", before=0.8140, after=0.8851, moved="noise \u2014 holdout machine is only ~60 rows"),
])
run2

,model,metric,before,after,moved
0,P1 Failure probability,ROC-AUC,0.9809,0.9911,no — see explanation below
1,P2 Failure type,F1 macro,0.6667,0.6884,"no, still rejected — consistent signal"
2,P3 RUL estimate,R²,0.9604,0.6661,YES — real generalization gap exposed
3,P5 Priority,F1 macro,0.8140,0.8851,noise — holdout machine is only ~60 rows


**P3 dropping from 96% to 67% is the split fix working.** P3's RUL estimate is computed relative to `train_df[tool_wear].max()` \u2014 the held-out machine has a different wear ceiling (each machine gets a random severity multiplier, 0.7\u20131.3\u00d7, in `seed_ml_data_all.py`), so a formula tuned to the training machines' scale doesn't transfer perfectly. That's a real, honest generalization gap.

**P1 barely moved, and here's precisely why the fix didn't reach it:** every machine was built from the exact same `wear >= 190 \u2192 TWF` rule, just rescaled. Holding out one machine only proves the model can apply one universal rule to a slightly-stretched copy of itself \u2014 it's never asked to guess, because there IS only one rule and all 4 machines obey it exactly. Group-holdout fixes Problem A (row leakage) but does nothing for Problem B (label = formula on the feature), and Problem B is what's driving P1's number.

**To actually fix P1**, one of:
1. Give each synthetic machine a genuinely different failure rule (not just a rescaled version of the same one), or
2. Add noise/randomness between the failure trigger and the label so it's not a hard deterministic cutoff, or
3. Stop relying on synthetic seed data for this number \u2014 wait for real technician-confirmed `actual_failure_type` values from real repairs. That data has genuine noise, confounders and ambiguity a formula can't fake.

## Part 4 — Bottom line, plain English

- **The retrain button now actually works**, safely (auto-restores on failure, gates bad models before they ship, doesn't silently corrupt files). That was **not true** before this session \u2014 it had never completed successfully even once.
- **P3's honest score (67%) is the one real signal in this batch** \u2014 it survived a genuine "never seen this machine before" test.
- **P2's rejection (69%, twice) is real too** \u2014 consistently below the bar, that's the model telling the truth about itself.
- **P1 and P5's high numbers are still not to be trusted.** Not because anything is broken now, but because the practice data I generated has the answer baked into the question for P1, and too small a test batch for P5's wiggle to mean anything.
- **P4 has no accuracy number by design** \u2014 it's an unsupervised anomaly detector, there's no labeled "correct answer" to score it against. A real evaluation would need to check whether its flags actually preceded real failures (precision/recall against outcomes) \u2014 not yet built.
- **P6/P7's displayed numbers are stale**, carried over from earlier training runs this session never touched or re-verified.

**Next step if a trustworthy P1 number matters:** either restructure the synthetic seed data so the label isn't a hard formula on the input feature, or wait for real repair data to accumulate and retrain on that instead.